## 1. Setup and Imports

In [ ]:
# Install required packages
%pip install neqsim pandas matplotlib numpy -q

import sys
sys.path.insert(0, '../../../python')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from neqsim.thermo.thermoTools import fluid, TPflash

from neqsim_fitting.data import load_density_data, get_data_summary
from neqsim_fitting.evaluation import calculate_all_metrics, parity_plot, deviation_plot
from neqsim_fitting.reporting import generate_fitting_report

In [ ]:
# Configuration
COMPONENT = 'hydrogen'
EQUATION_OF_STATE = 'srk'
DATA_FILE = '../../../data/thermodynamics/density/hydrogen.csv'

## 2. Load Experimental Data

In [ ]:
expdata_df = pd.read_csv(DATA_FILE)
print(f"Loaded {len(expdata_df)} data points")
expdata_df.head(10)

In [ ]:
# Data summary
print(f"Temperature range: {expdata_df['Temperature (K)'].min():.1f} - {expdata_df['Temperature (K)'].max():.1f} K")
print(f"Pressure range: {expdata_df['Pressure (bar)'].min():.1f} - {expdata_df['Pressure (bar)'].max():.1f} bar")
print(f"Density range: {expdata_df['Density (kg/m3)'].min():.4f} - {expdata_df['Density (kg/m3)'].max():.4f} kg/m³")

## 3. Data Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Density vs Pressure
ax1 = axes[0]
temps = expdata_df['Temperature (K)'].unique()
for temp in sorted(temps):
    data = expdata_df[expdata_df['Temperature (K)'] == temp]
    ax1.plot(data['Pressure (bar)'], data['Density (kg/m3)'], 'o-', label=f'{temp:.1f} K')
ax1.set_xlabel('Pressure (bar)')
ax1.set_ylabel('Density (kg/m³)')
ax1.set_title('Density vs Pressure')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Density vs Temperature
ax2 = axes[1]
pressures = [1, 50, 100, 200]
for pres in pressures:
    data = expdata_df[expdata_df['Pressure (bar)'].between(pres-5, pres+5)]
    if len(data) > 0:
        ax2.scatter(data['Temperature (K)'], data['Density (kg/m3)'], label=f'{pres} bar', alpha=0.7)
ax2.set_xlabel('Temperature (K)')
ax2.set_ylabel('Density (kg/m³)')
ax2.set_title('Density vs Temperature')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Model Comparison

In [ ]:
# Create fluid
fluid1 = fluid(EQUATION_OF_STATE)
fluid1.addComponent(COMPONENT, 1.0)

def calculate_density(temp, pres):
    """Calculate density at given T, P."""
    try:
        fluid1.setTemperature(temp, 'K')
        fluid1.setPressure(pres, 'bara')
        TPflash(fluid1)
        fluid1.initPhysicalProperties()
        return fluid1.getPhase(0).getPhysicalProperties().getDensity()
    except:
        return float('nan')

# Calculate for all data points
calculated_density = [
    calculate_density(row['Temperature (K)'], row['Pressure (bar)'])
    for _, row in expdata_df.iterrows()
]

expdata_df['density_calculated'] = calculated_density

In [ ]:
# Calculate metrics
metrics = calculate_all_metrics(
    expdata_df['Density (kg/m3)'].values,
    expdata_df['density_calculated'].values,
    name=EQUATION_OF_STATE.upper()
)

print(f"Model Performance ({EQUATION_OF_STATE.upper()}):")
print(f"  AAD%: {metrics['aad_percent']:.2f}%")
print(f"  RMSE: {metrics['rmse']:.4f} kg/m³")
print(f"  Max Error%: {metrics['max_error_percent']:.2f}%")
print(f"  Bias%: {metrics['bias_percent']:.2f}%")

In [ ]:
# Parity plot
parity_plot(
    expdata_df['Density (kg/m3)'].values,
    expdata_df['density_calculated'].values,
    title=f'{COMPONENT} Density Parity Plot ({EQUATION_OF_STATE.upper()})',
    xlabel='Experimental Density (kg/m³)',
    ylabel='Calculated Density (kg/m³)',
)

In [ ]:
# Deviation vs Pressure
deviation_plot(
    expdata_df['Pressure (bar)'].values,
    expdata_df['Density (kg/m3)'].values,
    expdata_df['density_calculated'].values,
    xlabel='Pressure (bar)',
    title='Density Deviation vs Pressure',
)

## 5. Compare Multiple Equations of State

In [ ]:
# Compare multiple EoS
eos_list = ['srk', 'pr', 'cpa']
results = {}

for eos in eos_list:
    try:
        fluid_test = fluid(eos)
        fluid_test.addComponent(COMPONENT, 1.0)
        
        calc_values = []
        for _, row in expdata_df.iterrows():
            try:
                fluid_test.setTemperature(row['Temperature (K)'], 'K')
                fluid_test.setPressure(row['Pressure (bar)'], 'bara')
                TPflash(fluid_test)
                fluid_test.initPhysicalProperties()
                calc_values.append(fluid_test.getPhase(0).getPhysicalProperties().getDensity())
            except:
                calc_values.append(float('nan'))
        
        results[eos.upper()] = calc_values
    except Exception as e:
        print(f"Error with {eos}: {e}")

# Calculate metrics for each
comparison_data = []
for eos_name, calc_vals in results.items():
    m = calculate_all_metrics(expdata_df['Density (kg/m3)'].values, np.array(calc_vals), name=eos_name)
    comparison_data.append(m)

comparison_df = pd.DataFrame(comparison_data).set_index('model_name')
comparison_df[['aad_percent', 'rmse', 'bias_percent', 'max_error_percent']]

## 6. Export Results

In [ ]:
# Calculate deviations
expdata_df['deviation'] = expdata_df['density_calculated'] - expdata_df['Density (kg/m3)']
expdata_df['deviation_percent'] = (expdata_df['deviation'] / expdata_df['Density (kg/m3)']) * 100

# Save results
output_file = f'./{COMPONENT}-density-comparison.csv'
expdata_df.to_csv(output_file, index=False)
print(f"Results saved to: {output_file}")

## 7. Summary

### Model Performance
| EoS | AAD% | RMSE | Bias% |
|-----|------|------|-------|
| [Model] | [Value] | [Value] | [Value] |

### Recommendations
- [Add your recommendations here]